# Sakila Lakehouse: one medallion layer at a time
**Databricks Free Edition · Delta Lake · Unity Catalog · Spark SQL / PySpark**

Builds a five-layer lakehouse from the Sakila CSVs, then loads a new month of data incrementally with deduplication, `MERGE`, Delta history and time travel. Every cell below was executed; outputs are from my run.

> Built in UChicago ADSP 31011 (Cloud-Native Data Engineering). The step-by-step scaffold was provided by the course; I wrote **Step 2.2** (Silver payment) and **Step 4.6** (deduplicated payment MERGE). User name and student ID are redacted. See the [project README](README.md) for a summary.

## Step 0 · Who ran this, and when
Delta records who ran every write; this is the identity that appears in the table history.

In [1]:
%sql
SELECT current_user() AS who_ran_this, current_timestamp() AS started_at

who_ran_this,started_at
<my-databricks-user>,2026-11-09T02:29:30.166Z


# Part 1 · Bronze: keep exactly what arrived
**Why Bronze exists:** it is the untouched copy of every file that arrived. If anything downstream goes wrong, you rebuild from here.
**One row is:** one line of a source file, every value still a string.

### Step 1.1 · Create the five layer schemas and the landing volume

In [2]:
%sql
USE CATALOG workspace;
CREATE SCHEMA IF NOT EXISTS sakila_bronze;
CREATE SCHEMA IF NOT EXISTS sakila_silver;
CREATE SCHEMA IF NOT EXISTS sakila_silver_enriched;
CREATE SCHEMA IF NOT EXISTS sakila_gold;
CREATE SCHEMA IF NOT EXISTS sakila_platinum;
CREATE VOLUME IF NOT EXISTS sakila_bronze.raw;

### Step 1.2 · Upload the files (no code)
1. Unzip **sakila_csv.zip** (Week 4 module, also in Week 6).
2. In **Catalog**, open workspace > sakila_bronze > raw, click **Upload to this volume**, and upload these 11 files:
   rental, payment, customer, film, inventory, category, film_category, store, address, city, country.
3. Keep **sakila_march2006.zip** for Part 4.

### Step 1.3 · Load every file into a Bronze table
`spark.read.csv` with a header and **no schema** keeps every column as a string. That is the point: Bronze fixes nothing.
Two audit columns are added: `_ingest_ts` (when it landed) and `_source_file` (which file it came from).
The write mode is **append**. Bronze is never updated or deleted.

⚠️ Run this cell **once**. Running it twice appends every file twice. (Silver would still dedupe it, but your Bronze counts would not match.)

In [3]:
from pyspark.sql.functions import current_timestamp, col

RAW = "/Volumes/workspace/sakila_bronze/raw/sakila_csv/"

def load_bronze(file_name, table):
    df = (spark.read.option("header", True).csv(f"{RAW}/{file_name}")
              .withColumn("_ingest_ts", current_timestamp())
              .withColumn("_source_file", col("_metadata.file_name")))
    df.write.mode("append").saveAsTable(f"sakila_bronze.{table}")
    print(f"{table:<22} appended {df.count():>6,} rows from {file_name}")

for t in ["rental", "payment", "customer", "film", "inventory", "category",
          "film_category", "store", "address", "city", "country"]:
    load_bronze(f"{t}.csv", f"{t}_raw")

rental_raw             appended 16,044 rows from rental.csv
payment_raw            appended 16,049 rows from payment.csv
customer_raw           appended    599 rows from customer.csv
film_raw               appended  1,000 rows from film.csv
inventory_raw          appended  4,581 rows from inventory.csv
category_raw           appended     16 rows from category.csv
film_category_raw      appended  1,000 rows from film_category.csv
store_raw              appended      2 rows from store.csv
address_raw            appended    603 rows from address.csv
city_raw               appended    600 rows from city.csv
country_raw            appended    109 rows from country.csv


### Step 1.4 · Bronze row counts
Expected: rental_raw **16,044** · payment_raw **16,049** · customer_raw **599** · film_raw **1,000** · inventory_raw **4,581**.

In [4]:
def layer_counts(schema):
    tables = [r.tableName for r in spark.sql(f"SHOW TABLES IN workspace.{schema}").collect()]
    q = " UNION ALL ".join(f"SELECT '{t}' AS table_name, COUNT(*) AS row_count FROM workspace.{schema}.{t}" for t in sorted(tables))
    return spark.sql(q)

display(layer_counts("sakila_bronze"))

table_name,row_count
address_raw,603
category_raw,16
city_raw,600
country_raw,109
customer_raw,599
film_category_raw,1000
film_raw,1000
inventory_raw,4581
payment_raw,16049
rental_raw,16044


**Look at a Bronze table.** Every source column is a string; only the two audit columns are typed. (address_raw is worse: the MySQL export left a geometry value in an extra field, so its last_update column holds junk. Bronze keeps it anyway, and Silver drops it in Step 2.4.)

In [5]:
%sql
DESCRIBE TABLE workspace.sakila_bronze.rental_raw

col_name,data_type,comment
rental_id,string,None
rental_date,string,None
inventory_id,string,None
customer_id,string,None
return_date,string,None
staff_id,string,None
last_update,string,None
_ingest_ts,timestamp,None
_source_file,string,None
# Clustering Information,,


# Part 2 · Silver: clean, type and dedupe
**Why Silver exists:** downstream layers need one trusted row per key, with real types. That cleaning happens once, here.
**One row is:** one rental, one payment, one customer, and so on.

### Step 2.1 · Silver rental
* `CAST` turns strings into real types.
* `QUALIFY ROW_NUMBER() ... = 1` keeps one row per rental_id (the latest arrival) and is the dedupe.
* Rentals with no return_date are real (the movie is still out), so they are kept.

In [6]:
%sql
USE CATALOG workspace;
CREATE OR REPLACE TABLE sakila_silver.rental AS
SELECT CAST(rental_id    AS INT)       AS rental_id,
       CAST(rental_date  AS TIMESTAMP) AS rental_date,
       CAST(inventory_id AS INT)       AS inventory_id,
       CAST(customer_id  AS INT)       AS customer_id,
       CAST(NULLIF(TRIM(return_date), '') AS TIMESTAMP) AS return_date,
       CAST(staff_id     AS INT)       AS staff_id,
       CAST(last_update  AS TIMESTAMP) AS last_update,
       _ingest_ts
FROM sakila_bronze.rental_raw
QUALIFY ROW_NUMBER() OVER (PARTITION BY rental_id ORDER BY _ingest_ts DESC) = 1;

num_affected_rows,num_inserted_rows


### Step 2.2 · Silver payment
Same pattern as the rental cell above, applied to `sakila_bronze.payment_raw` → `sakila_silver.payment`.
Columns: payment_id, customer_id, staff_id, rental_id (INT); amount (**DECIMAL(5,2)**); payment_date, last_update (TIMESTAMP); _ingest_ts. Dedupe on payment_id.
Keep the 5 payments with no rental_id: they are real money. Check: **16,049** rows.

In [7]:
%sql
USE CATALOG workspace;
-- Silver payment: type, keep NULL rental_ids, dedupe on payment_id
CREATE OR REPLACE TABLE sakila_silver.payment AS
SELECT 
       CAST(payment_id AS INT) AS payment_id,
       CAST(customer_id AS INT) AS customer_id,
       CAST(staff_id AS INT) AS staff_id,
         CAST(NULLIF(TRIM(rental_id), '') AS INT) AS rental_id,
       CAST(amount AS DECIMAL(5,2)) AS amount,
      CAST(payment_date AS TIMESTAMP) AS payment_date,    
       CAST(last_update  AS TIMESTAMP) AS last_update,
       _ingest_ts
FROM sakila_bronze.payment_raw
QUALIFY ROW_NUMBER() OVER (PARTITION BY payment_id ORDER BY _ingest_ts DESC) = 1;

num_affected_rows,num_inserted_rows


In [8]:
%sql
SELECT COUNT(*) FROM workspace.sakila_silver.payment

COUNT(*)
16049


### Step 2.3 · Silver customer and film
Conform as well as cast: names in Title Case, emails in lower case, and `active` as a real boolean.

In [9]:
%sql
USE CATALOG workspace;
CREATE OR REPLACE TABLE sakila_silver.customer AS
SELECT CAST(customer_id AS INT)       AS customer_id,
       CAST(store_id    AS INT)       AS store_id,
       INITCAP(TRIM(first_name))      AS first_name,
       INITCAP(TRIM(last_name))       AS last_name,
       LOWER(TRIM(email))             AS email,
       CAST(address_id  AS INT)       AS address_id,
       CAST(active AS INT) = 1        AS active,
       CAST(create_date AS TIMESTAMP) AS create_date,
       _ingest_ts
FROM sakila_bronze.customer_raw
QUALIFY ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY _ingest_ts DESC) = 1;

CREATE OR REPLACE TABLE sakila_silver.film AS
SELECT CAST(film_id AS INT)                   AS film_id,
       TRIM(title)                            AS title,
       CAST(release_year AS INT)              AS release_year,
       CAST(rental_duration AS INT)           AS rental_duration,
       CAST(rental_rate AS DECIMAL(4,2))      AS rental_rate,
       CAST(length AS INT)                    AS length_minutes,
       CAST(replacement_cost AS DECIMAL(5,2)) AS replacement_cost,
       rating,
       _ingest_ts
FROM sakila_bronze.film_raw
QUALIFY ROW_NUMBER() OVER (PARTITION BY film_id ORDER BY _ingest_ts DESC) = 1;

num_affected_rows,num_inserted_rows


In [10]:
%sql
SELECT (SELECT COUNT(*) FROM workspace.sakila_silver.customer) AS customers,
       (SELECT COUNT(*) FROM workspace.sakila_silver.film)     AS films

customers,films
599,1000


### Step 2.4 · The small lookup tables
address drops the junk `last_update` field you saw in Bronze. That is a cleaning decision, and Silver is where it belongs.

In [11]:
%sql
USE CATALOG workspace;
CREATE OR REPLACE TABLE sakila_silver.inventory AS
SELECT DISTINCT CAST(inventory_id AS INT) AS inventory_id, CAST(film_id AS INT) AS film_id, CAST(store_id AS INT) AS store_id
FROM sakila_bronze.inventory_raw;

CREATE OR REPLACE TABLE sakila_silver.category AS
SELECT DISTINCT CAST(category_id AS INT) AS category_id, TRIM(name) AS category FROM sakila_bronze.category_raw;

CREATE OR REPLACE TABLE sakila_silver.film_category AS
SELECT DISTINCT CAST(film_id AS INT) AS film_id, CAST(category_id AS INT) AS category_id FROM sakila_bronze.film_category_raw;

CREATE OR REPLACE TABLE sakila_silver.store AS
SELECT DISTINCT CAST(store_id AS INT) AS store_id, CAST(address_id AS INT) AS address_id FROM sakila_bronze.store_raw;

CREATE OR REPLACE TABLE sakila_silver.address AS
SELECT DISTINCT CAST(address_id AS INT) AS address_id, TRIM(address) AS address, TRIM(district) AS district,
       CAST(city_id AS INT) AS city_id
FROM sakila_bronze.address_raw;

CREATE OR REPLACE TABLE sakila_silver.city AS
SELECT DISTINCT CAST(city_id AS INT) AS city_id, TRIM(city) AS city, CAST(country_id AS INT) AS country_id FROM sakila_bronze.city_raw;

CREATE OR REPLACE TABLE sakila_silver.country AS
SELECT DISTINCT CAST(country_id AS INT) AS country_id, TRIM(country) AS country FROM sakila_bronze.country_raw;

num_affected_rows,num_inserted_rows


### Step 2.5 · Silver row counts
Expected: rental **16,044** · payment **16,049** · customer **599** · film **1,000** · inventory **4,581**.

In [12]:
display(layer_counts("sakila_silver"))

table_name,row_count
address,603
category,16
city,600
country,109
customer,599
film,1000
film_category,1000
inventory,4581
payment,16049
rental,16044


### Step 2.6 · Get to know Sakila
Silver is clean enough to explore. Four quick questions:

**(a) Rentals by month.** Expect 2005-05 through 2005-08, plus 182 rentals in 2006-02.

In [13]:
%sql
SELECT DATE_FORMAT(rental_date, 'yyyy-MM') AS rental_month, COUNT(*) AS rentals
FROM workspace.sakila_silver.rental
GROUP BY rental_month ORDER BY rental_month

rental_month,rentals
2005-05,1156
2005-06,2311
2005-07,6709
2005-08,5686
2006-02,182


**(b) The 10 most-rented films.** BUCKET BROTHERHOOD leads with 34.

In [14]:
%sql
SELECT f.title, COUNT(*) AS rentals
FROM workspace.sakila_silver.rental r
JOIN workspace.sakila_silver.inventory i ON i.inventory_id = r.inventory_id
JOIN workspace.sakila_silver.film f      ON f.film_id = i.film_id
GROUP BY f.title ORDER BY rentals DESC, f.title LIMIT 10

title,rentals
BUCKET BROTHERHOOD,34
ROCKETEER MOTHER,33
FORWARD TEMPLE,32
GRIT CLOCKWORK,32
JUGGLER HARDLY,32
RIDGEMONT SUBMARINE,32
SCALAWAG DUCK,32
APACHE DIVINE,31
GOODFELLAS SALUTE,31
HOBBIT ALIEN,31


**(c) The 5 countries with the most customers.** India (60) and China (53) lead.

In [15]:
%sql
SELECT co.country, COUNT(*) AS customers
FROM workspace.sakila_silver.customer cu
JOIN workspace.sakila_silver.address a ON a.address_id = cu.address_id
JOIN workspace.sakila_silver.city ci   ON ci.city_id = a.city_id
JOIN workspace.sakila_silver.country co ON co.country_id = ci.country_id
GROUP BY co.country ORDER BY customers DESC LIMIT 5

country,customers
India,60
China,53
United States,36
Japan,31
Mexico,30


**(d) Total payments, and the payments with no rental.** Expect 67,416.51 in total, and 5 payments (9.95) with a null rental_id.

In [16]:
%sql
SELECT SUM(amount) AS total_payments,
       COUNT_IF(rental_id IS NULL) AS payments_without_rental,
       SUM(CASE WHEN rental_id IS NULL THEN amount END) AS amount_without_rental
FROM workspace.sakila_silver.payment

total_payments,payments_without_rental,amount_without_rental
67416.51,5,9.95


# Part 3 · Silver-Enriched and Gold
### Step 3.1 · Silver-Enriched: one wide row per rental
**Why Silver-Enriched exists:** most questions need the rental *and* its film, category, customer, country and payment. The joins are done once, here, so nobody downstream repeats them or gets them wrong.
**One row is:** one rental, with everything about it.

Two details matter:
* Payments are **summed per rental first** and then left-joined. Joining payment rows directly could multiply rentals.
* `is_late` is NULL while a rental is still out, because it isn't late or on time yet.

In [17]:
%sql
USE CATALOG workspace;
CREATE OR REPLACE TABLE sakila_silver_enriched.rental_enriched AS
WITH paid AS (
  SELECT rental_id, SUM(amount) AS amount_paid
  FROM sakila_silver.payment WHERE rental_id IS NOT NULL GROUP BY rental_id
)
SELECT r.rental_id, r.rental_date, r.return_date,
       DATE_FORMAT(r.rental_date, 'yyyy-MM') AS rental_month,
       r.customer_id, cu.first_name, cu.last_name,
       ci.city, co.country,
       i.store_id, f.film_id, f.title, cat.category, f.rating, f.rental_duration,
       p.amount_paid,
       DATEDIFF(CAST(r.return_date AS DATE), CAST(r.rental_date AS DATE)) AS rental_days,
       CASE WHEN r.return_date IS NULL THEN NULL
            ELSE DATEDIFF(CAST(r.return_date AS DATE), CAST(r.rental_date AS DATE)) > f.rental_duration END AS is_late
FROM sakila_silver.rental r
JOIN sakila_silver.inventory i      ON i.inventory_id = r.inventory_id
JOIN sakila_silver.film f           ON f.film_id = i.film_id
JOIN sakila_silver.film_category fc ON fc.film_id = f.film_id
JOIN sakila_silver.category cat     ON cat.category_id = fc.category_id
JOIN sakila_silver.customer cu      ON cu.customer_id = r.customer_id
JOIN sakila_silver.address a        ON a.address_id = cu.address_id
JOIN sakila_silver.city ci          ON ci.city_id = a.city_id
JOIN sakila_silver.country co       ON co.country_id = ci.country_id
LEFT JOIN paid p                    ON p.rental_id = r.rental_id;

num_affected_rows,num_inserted_rows


### Step 3.2 · Check Silver-Enriched
The row counts must be equal. More rows would mean a join fanned out; fewer would mean a join dropped rentals.
Expected: **16,044 = 16,044**, and revenue **67,406.56**, which is 9.95 less than total payments. Those are the 5 payments with no rental: a rental-level table can't hold them.

In [18]:
%sql
SELECT (SELECT COUNT(*) FROM workspace.sakila_silver.rental)                    AS silver_rentals,
       (SELECT COUNT(*) FROM workspace.sakila_silver_enriched.rental_enriched)   AS enriched_rentals,
       (SELECT SUM(amount_paid) FROM workspace.sakila_silver_enriched.rental_enriched) AS enriched_revenue,
       (SELECT SUM(amount) FROM workspace.sakila_silver.payment)                 AS all_payments

silver_rentals,enriched_rentals,enriched_revenue,all_payments
16044,16044,67406.56,67416.51


### Step 3.3 · Gold: a star schema
**Why Gold exists:** analysts and BI tools want a model they can reason about: one fact table of events, with dimensions to slice by.
**One row is:** in the fact table, one rental; in the dimensions, one customer, film, store or day.

In [19]:
%sql
USE CATALOG workspace;
CREATE OR REPLACE TABLE sakila_gold.fact_rental AS
SELECT rental_id, customer_id, film_id, store_id,
       CAST(rental_date AS DATE) AS date_key,
       rental_date, return_date, rental_month,
       amount_paid, rental_days, is_late
FROM sakila_silver_enriched.rental_enriched;

CREATE OR REPLACE TABLE sakila_gold.dim_customer AS
SELECT cu.customer_id, cu.first_name, cu.last_name, cu.email, cu.active, cu.store_id, ci.city, co.country
FROM sakila_silver.customer cu
JOIN sakila_silver.address a  ON a.address_id = cu.address_id
JOIN sakila_silver.city ci    ON ci.city_id = a.city_id
JOIN sakila_silver.country co ON co.country_id = ci.country_id;

CREATE OR REPLACE TABLE sakila_gold.dim_film AS
SELECT f.film_id, f.title, cat.category, f.rating, f.rental_rate, f.rental_duration, f.length_minutes
FROM sakila_silver.film f
JOIN sakila_silver.film_category fc ON fc.film_id = f.film_id
JOIN sakila_silver.category cat     ON cat.category_id = fc.category_id;

CREATE OR REPLACE TABLE sakila_gold.dim_store AS
SELECT s.store_id, ci.city AS store_city, co.country AS store_country
FROM sakila_silver.store s
JOIN sakila_silver.address a  ON a.address_id = s.address_id
JOIN sakila_silver.city ci    ON ci.city_id = a.city_id
JOIN sakila_silver.country co ON co.country_id = ci.country_id;

CREATE OR REPLACE TABLE sakila_gold.dim_date AS
SELECT d AS date_key, YEAR(d) AS year, MONTH(d) AS month, DATE_FORMAT(d, 'yyyy-MM') AS year_month,
       DATE_FORMAT(d, 'EEEE') AS day_name
FROM (SELECT EXPLODE(SEQUENCE(MIN(CAST(rental_date AS DATE)), MAX(CAST(rental_date AS DATE)), INTERVAL 1 DAY)) AS d
      FROM sakila_silver.rental);

num_affected_rows,num_inserted_rows


### Step 3.4 · Gold business aggregate: revenue by category by month

In [20]:
%sql
USE CATALOG workspace;
CREATE OR REPLACE TABLE sakila_gold.revenue_by_category_month AS
SELECT f.rental_month, d.category, COUNT(*) AS rentals, SUM(f.amount_paid) AS revenue
FROM sakila_gold.fact_rental f
JOIN sakila_gold.dim_film d ON d.film_id = f.film_id
GROUP BY f.rental_month, d.category;

SELECT * FROM sakila_gold.revenue_by_category_month ORDER BY revenue DESC LIMIT 5;

rental_month,category,rentals,revenue
2005-07,Sports,497,2281.03
2005-07,Sci-Fi,462,2013.38
2005-07,Drama,463,1978.37
2005-08,Sports,432,1962.68
2005-07,Animation,489,1954.11


### Step 3.5 · Spot-check one film
Picks one film by ID and reports its rentals and revenue; re-run after the March load to see it change.

In [21]:
my_film = 79   # BLADE POLISH
display(spark.sql(f'''
  SELECT d.film_id, d.title, d.category, COUNT(*) AS rentals, SUM(f.amount_paid) AS revenue
  FROM workspace.sakila_gold.fact_rental f JOIN workspace.sakila_gold.dim_film d ON d.film_id = f.film_id
  WHERE d.film_id = {my_film} GROUP BY d.film_id, d.title, d.category'''))

film_id,title,category,rentals,revenue
79,BLADE POLISH,Drama,26,38.75


### Step 3.6 · Save the "before" row counts
This records the row count of the main table at every layer, so Part 4 can show what the new data changed.

In [22]:
from pyspark.sql.functions import lit

MAIN = [("1 Bronze", "sakila_bronze.rental_raw"), ("1 Bronze", "sakila_bronze.payment_raw"),
        ("2 Silver", "sakila_silver.rental"), ("2 Silver", "sakila_silver.payment"),
        ("3 Silver-Enriched", "sakila_silver_enriched.rental_enriched"),
        ("4 Gold", "sakila_gold.fact_rental"), ("4 Gold", "sakila_gold.dim_date"),
        ("4 Gold", "sakila_gold.revenue_by_category_month")]

def snapshot(label):
    q = " UNION ALL ".join(f"SELECT '{layer}' AS layer, '{t}' AS table_name, COUNT(*) AS row_count FROM workspace.{t}" for layer, t in MAIN)
    spark.sql(q).withColumn("snapshot", lit(label)).write.mode("append").saveAsTable("workspace.default.layer_counts")

snapshot("before")
display(spark.table("workspace.default.layer_counts"))

layer,table_name,row_count,snapshot
1 Bronze,sakila_bronze.rental_raw,16044,before
1 Bronze,sakila_bronze.payment_raw,16049,before
2 Silver,sakila_silver.rental,16044,before
2 Silver,sakila_silver.payment,16049,before
3 Silver-Enriched,sakila_silver_enriched.rental_enriched,16044,before
4 Gold,sakila_gold.fact_rental,16044,before
4 Gold,sakila_gold.dim_date,267,before
4 Gold,sakila_gold.revenue_by_category_month,80,before


# Part 4 · Salt in a new month of data
A month later, the source system sends more. Unzip **sakila_march2006.zip** and upload its three files to the same raw volume:
* **rental_2006_03.csv** (2,575 rows): 2,400 new March 2006 rentals, 150 February rentals that have now been returned (updates to rows you already have), and 25 rentals the source sent twice.
* **payment_2006_03.csv** (2,415 rows): one payment per new rental, plus 15 re-sent rows.
* **country_region.csv** (109 rows): a new reference file that maps each country to a region.

### Step 4.1 · Bronze: append the new files with the same loader
Nothing about Bronze changes: same function, append only. `_source_file` tells the batches apart.

In [23]:
load_bronze("rental_2006_03.csv",  "rental_raw")
load_bronze("payment_2006_03.csv", "payment_raw")
load_bronze("country_region.csv",  "country_region_raw")

rental_raw             appended  2,575 rows from rental_2006_03.csv
payment_raw            appended  2,415 rows from payment_2006_03.csv
country_region_raw     appended    109 rows from country_region.csv


### Step 4.2 · Bronze after the salt-in
Expected: rental_raw **18,619** (16,044 + 2,575) · payment_raw **18,464** · country_region_raw **109**.

In [24]:
%sql
SELECT _source_file, COUNT(*) AS rows_in_bronze
FROM workspace.sakila_bronze.rental_raw GROUP BY _source_file
UNION ALL
SELECT _source_file, COUNT(*) FROM workspace.sakila_bronze.payment_raw GROUP BY _source_file
ORDER BY _source_file

_source_file,rows_in_bronze
payment.csv,16049
payment_2006_03.csv,2415
rental.csv,16044
rental_2006_03.csv,2575


### Step 4.3 · The March rentals, cleaned the Silver way, but NOT deduped yet
A temporary view with the same casts as Step 2.1, reading only the March file.

In [25]:
%sql
CREATE OR REPLACE TEMP VIEW rental_march_raw AS
SELECT CAST(rental_id    AS INT)       AS rental_id,
       CAST(rental_date  AS TIMESTAMP) AS rental_date,
       CAST(inventory_id AS INT)       AS inventory_id,
       CAST(customer_id  AS INT)       AS customer_id,
       CAST(NULLIF(TRIM(return_date), '') AS TIMESTAMP) AS return_date,
       CAST(staff_id     AS INT)       AS staff_id,
       CAST(last_update  AS TIMESTAMP) AS last_update,
       _ingest_ts
FROM workspace.sakila_bronze.rental_raw
WHERE _source_file = 'rental_2006_03.csv';

SELECT COUNT(*) AS rows, COUNT(DISTINCT rental_id) AS distinct_rentals FROM rental_march_raw;

rows,distinct_rentals
2575,2550


### Step 4.4 · Break it on purpose: MERGE without deduping
**This cell is supposed to fail.** The error shows why duplicates must be removed before a MERGE.
Ten of the duplicated rentals match rows already in Silver. MERGE refuses to update one target row from two source rows, because it can't know which one is right.

In [26]:
%sql
MERGE INTO workspace.sakila_silver.rental AS t
USING rental_march_raw AS s
ON t.rental_id = s.rental_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

UnsupportedOperationException: [DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW_IN_MERGE] Cannot perform Merge as multiple source rows matched and attempted to modify the same

### Step 4.5 · Dedupe, then MERGE
This is the same `QUALIFY` dedupe as Step 2.1. Then MERGE: **update** the rentals Silver already has (the February returns) and **insert** the new ones.
Expected: **2,550** source rows (2,400 new + 150 updates), and Silver rental goes from 16,044 to **18,444**.

In [27]:
%sql
CREATE OR REPLACE TEMP VIEW rental_march AS
SELECT * FROM rental_march_raw
QUALIFY ROW_NUMBER() OVER (PARTITION BY rental_id ORDER BY _ingest_ts DESC) = 1;

MERGE INTO workspace.sakila_silver.rental AS t
USING rental_march AS s
ON t.rental_id = s.rental_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2550,150,0,2400


### Step 4.6 · MERGE the March payments
Same pattern as Step 4.5, applied to payments: a view over `payment_raw` where `_source_file = 'payment_2006_03.csv'`, with the Step 2.2 casts, deduped on payment_id, then MERGE into `sakila_silver.payment` on payment_id.
Expected: 2,400 inserted, 0 updated; Silver payment **18,449**.

In [28]:
%sql
-- Deduplicated view of the March payments, then MERGE into Silver payment
CREATE OR REPLACE TEMP VIEW payment_march AS
SELECT
  CAST(payment_id AS INT) AS payment_id,
  CAST(customer_id AS INT) AS customer_id,
  CAST(staff_id AS INT) AS staff_id,
  CAST(NULLIF(TRIM(rental_id), '') AS INT) AS rental_id,
  CAST(amount AS DECIMAL(5,2)) AS amount,
  CAST(payment_date AS TIMESTAMP) AS payment_date,
  CAST(last_update AS TIMESTAMP) AS last_update,
  _ingest_ts
FROM workspace.sakila_bronze.payment_raw
WHERE _source_file = 'payment_2006_03.csv'
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY payment_id
  ORDER BY _ingest_ts DESC
) = 1;

MERGE INTO workspace.sakila_silver.payment AS t
USING payment_march AS s
ON t.payment_id = s.payment_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2400,0,0,2400


### Step 4.7 · Read the Delta history
Every write to a Delta table is a new **version**: who did it, when, which operation, and what changed.
Find the MERGE row. Its `operationMetrics` should show **numTargetRowsInserted = 2400** and **numTargetRowsUpdated = 150**.

In [29]:
display(spark.sql("DESCRIBE HISTORY workspace.sakila_silver.rental")
        .select("version", "timestamp", "userName", "operation", "operationMetrics"))

version,timestamp,userName,operation,operationMetrics
3,2026-11-09T03:13:59.000Z,<my-databricks-user>,MERGE,"{""numTargetRowsCopied"": ""0"", ""numTargetRowsDeleted"": ""0"", ""numTargetFilesAdded"": ""1"", ""numTargetBytesAdded"": ""36842"", ""numTargetBytesRemoved"": ""0"", ""numTargetDeletionVectorsAdded"": ""1"", ""numTargetRowsMatchedUpdated"": ""150"", ""executionTimeMs"": ""2772"", ""materializeSourceTimeMs"": ""352"", ""numTargetRowsInserted"": ""2400"", ""numTargetRowsMatchedDeleted"": ""0"", ""numTargetDeletionVectorsUpdated"": ""0"", ""scanTimeMs"": ""636"", ""numTargetRowsUpdated"": ""150"", ""numOutputRows"": ""2550"", ""numTargetDeletionVectorsRemoved"": ""0"", ""numTargetRowsNotMatchedBySourceUpdated"": ""0"", ""numTargetChangeFilesAdded"": ""0"", ""numSourceRows"": ""2550"", ""numTargetFilesRemoved"": ""0"", ""numTargetRowsNotMatchedBySourceDeleted"": ""0"", ""rewriteTimeMs"": ""1729""}"
2,2026-11-09T02:52:22.000Z,<my-databricks-user>,CREATE OR REPLACE TABLE AS SELECT,"{""numFiles"": ""1"", ""numRemovedFiles"": ""1"", ""numRemovedBytes"": ""229269"", ""numDeletionVectorsRemoved"": ""0"", ""numOutputRows"": ""16044"", ""numOutputBytes"": ""229269""}"
1,2026-11-09T02:46:15.000Z,<my-databricks-user>,CREATE OR REPLACE TABLE AS SELECT,"{""numFiles"": ""1"", ""numRemovedFiles"": ""1"", ""numRemovedBytes"": ""229269"", ""numDeletionVectorsRemoved"": ""0"", ""numOutputRows"": ""16044"", ""numOutputBytes"": ""229269""}"
0,2026-11-09T02:43:05.000Z,<my-databricks-user>,CREATE OR REPLACE TABLE AS SELECT,"{""numFiles"": ""1"", ""numRemovedFiles"": ""0"", ""numRemovedBytes"": ""0"", ""numDeletionVectorsRemoved"": ""0"", ""numOutputRows"": ""16044"", ""numOutputBytes"": ""229269""}"


### Step 4.8 · The Delta difference: before and after the MERGE
Time travel reads the table as it was at an earlier version. The cell finds the version just before the MERGE and compares it with now.
Expected: **16,044** before, **18,444** now, **2,550** rows new or changed, and **150** rentals that were open before and are returned now.

In [30]:
from pyspark.sql.functions import min as smin
h = spark.sql("DESCRIBE HISTORY workspace.sakila_silver.rental")
before = h.filter("operation = 'MERGE'").agg(smin("version")).first()[0] - 1
print("Version before the MERGE:", before)

display(spark.sql(f'''
  SELECT (SELECT COUNT(*) FROM workspace.sakila_silver.rental VERSION AS OF {before}) AS rows_before,
         (SELECT COUNT(*) FROM workspace.sakila_silver.rental)                       AS rows_now,
         (SELECT COUNT(*) FROM (SELECT * FROM workspace.sakila_silver.rental
                                EXCEPT
                                SELECT * FROM workspace.sakila_silver.rental VERSION AS OF {before})) AS rows_new_or_changed'''))

display(spark.sql(f'''
  SELECT n.rental_id, o.return_date AS return_date_before, n.return_date AS return_date_now
  FROM workspace.sakila_silver.rental n
  JOIN workspace.sakila_silver.rental VERSION AS OF {before} o ON o.rental_id = n.rental_id
  WHERE o.return_date IS NULL AND n.return_date IS NOT NULL
  ORDER BY n.rental_id'''))

Version before the MERGE: 2


rows_before,rows_now,rows_new_or_changed
16044,18444,2550


rental_id,return_date_before,return_date_now
11496,None,2006-03-07T04:12:00.000Z
11541,None,2006-03-09T08:25:00.000Z
11563,None,2006-03-11T02:10:00.000Z
11577,None,2006-03-17T16:14:00.000Z
11611,None,2006-03-15T21:43:00.000Z
11646,None,2006-03-10T08:41:00.000Z
11652,None,2006-03-10T17:46:00.000Z
11657,None,2006-03-16T02:36:00.000Z
11672,None,2006-03-06T17:59:00.000Z
11676,None,2006-03-07T13:58:00.000Z


### Step 4.9 · The new reference data, in Silver

In [31]:
%sql
USE CATALOG workspace;
CREATE OR REPLACE TABLE sakila_silver.country_region AS
SELECT DISTINCT CAST(country_id AS INT) AS country_id, TRIM(country) AS country, TRIM(region) AS region
FROM sakila_bronze.country_region_raw;

num_affected_rows,num_inserted_rows


### Step 4.10 · Re-run Silver-Enriched and Gold, now with region
This is the same statement as Step 3.1, with **one new join** and **one new column** (`region`). `CREATE OR REPLACE` rebuilds the table from the new Silver.
After this cell, **re-run Steps 3.3 and 3.4 unchanged**, then run the next cell to add region to dim_customer.

In [32]:
%sql
USE CATALOG workspace;
CREATE OR REPLACE TABLE sakila_silver_enriched.rental_enriched AS
WITH paid AS (
  SELECT rental_id, SUM(amount) AS amount_paid
  FROM sakila_silver.payment WHERE rental_id IS NOT NULL GROUP BY rental_id
)
SELECT r.rental_id, r.rental_date, r.return_date,
       DATE_FORMAT(r.rental_date, 'yyyy-MM') AS rental_month,
       r.customer_id, cu.first_name, cu.last_name,
       ci.city, co.country, cr.region,
       i.store_id, f.film_id, f.title, cat.category, f.rating, f.rental_duration,
       p.amount_paid,
       DATEDIFF(CAST(r.return_date AS DATE), CAST(r.rental_date AS DATE)) AS rental_days,
       CASE WHEN r.return_date IS NULL THEN NULL
            ELSE DATEDIFF(CAST(r.return_date AS DATE), CAST(r.rental_date AS DATE)) > f.rental_duration END AS is_late
FROM sakila_silver.rental r
JOIN sakila_silver.inventory i      ON i.inventory_id = r.inventory_id
JOIN sakila_silver.film f           ON f.film_id = i.film_id
JOIN sakila_silver.film_category fc ON fc.film_id = f.film_id
JOIN sakila_silver.category cat     ON cat.category_id = fc.category_id
JOIN sakila_silver.customer cu      ON cu.customer_id = r.customer_id
JOIN sakila_silver.address a        ON a.address_id = cu.address_id
JOIN sakila_silver.city ci          ON ci.city_id = a.city_id
JOIN sakila_silver.country co       ON co.country_id = ci.country_id
LEFT JOIN sakila_silver.country_region cr ON cr.country_id = co.country_id   -- NEW
LEFT JOIN paid p                    ON p.rental_id = r.rental_id;

num_affected_rows,num_inserted_rows


**Re-run Step 3.3 and Step 3.4 now**, then run this cell.

In [33]:
%sql
USE CATALOG workspace;
CREATE OR REPLACE TABLE sakila_gold.dim_customer AS
SELECT cu.customer_id, cu.first_name, cu.last_name, cu.email, cu.active, cu.store_id, ci.city, co.country, cr.region
FROM sakila_silver.customer cu
JOIN sakila_silver.address a  ON a.address_id = cu.address_id
JOIN sakila_silver.city ci    ON ci.city_id = a.city_id
JOIN sakila_silver.country co ON co.country_id = ci.country_id
LEFT JOIN sakila_silver.country_region cr ON cr.country_id = co.country_id;

SELECT COUNT(*) AS rentals, SUM(amount_paid) AS revenue, COUNT_IF(region IS NULL) AS rentals_without_region
FROM sakila_silver_enriched.rental_enriched;

rentals,revenue,rentals_without_region
18444,77099.56,0


### Step 4.11 · Before and after, at every layer
Then re-run **Step 3.5** (the film spot-check).

In [34]:
snapshot("after")
display(spark.sql('''
  SELECT layer, table_name,
         MAX(CASE WHEN snapshot = 'before' THEN row_count END) AS before,
         MAX(CASE WHEN snapshot = 'after'  THEN row_count END) AS after,
         MAX(CASE WHEN snapshot = 'after'  THEN row_count END) - MAX(CASE WHEN snapshot = 'before' THEN row_count END) AS change
  FROM workspace.default.layer_counts
  GROUP BY layer, table_name ORDER BY layer, table_name'''))

layer,table_name,before,after,change
1 Bronze,sakila_bronze.payment_raw,16049,18464,2415
1 Bronze,sakila_bronze.rental_raw,16044,18619,2575
2 Silver,sakila_silver.payment,16049,18449,2400
2 Silver,sakila_silver.rental,16044,18444,2400
3 Silver-Enriched,sakila_silver_enriched.rental_enriched,16044,18444,2400
4 Gold,sakila_gold.dim_date,267,312,45
4 Gold,sakila_gold.fact_rental,16044,18444,2400
4 Gold,sakila_gold.revenue_by_category_month,80,96,16


# Part 5 · Platinum: serve it
**Why Platinum exists:** a dashboard should read one simple, documented view, not rebuild joins and business rules itself.
**One row is:** one point on a chart.

### Step 5.1 · Monthly revenue view

In [35]:
%sql
USE CATALOG workspace;
CREATE OR REPLACE VIEW sakila_platinum.monthly_revenue (
  month    COMMENT 'Rental month, yyyy-MM',
  rentals  COMMENT 'Rentals that started in the month',
  revenue  COMMENT 'Payments for those rentals, counted in the month of the rental',
  late_pct COMMENT 'Late rentals as a % of returned rentals (open rentals excluded)'
) COMMENT 'One row per rental month. Reads sakila_gold.fact_rental.'
AS SELECT rental_month AS month, COUNT(*) AS rentals, SUM(amount_paid) AS revenue,
          ROUND(100 * COUNT_IF(is_late) / NULLIF(COUNT(return_date), 0), 1) AS late_pct
   FROM sakila_gold.fact_rental
   GROUP BY rental_month;

SELECT * FROM sakila_platinum.monthly_revenue ORDER BY month;

month,rentals,revenue,late_pct
2005-05,1156,4823.44,44.6
2005-06,2311,9629.89,45.9
2005-07,6709,28368.91,45.8
2005-08,5686,24070.14,46.1
2006-02,182,514.18,100.0
2006-03,2400,9693.00,48.5


### Step 5.2 · Revenue by region and category view
Expected: Asia is the top region (34,986.68).

In [36]:
%sql
USE CATALOG workspace;
CREATE OR REPLACE VIEW sakila_platinum.revenue_by_region_category (
  region   COMMENT 'Customer region, from country_region.csv',
  category COMMENT 'Film category',
  rentals  COMMENT 'Rentals by customers in the region, of films in the category',
  revenue  COMMENT 'Payments for those rentals'
) COMMENT 'One row per region and category. Reads sakila_gold.fact_rental, dim_customer and dim_film.'
AS SELECT c.region, d.category, COUNT(*) AS rentals, SUM(f.amount_paid) AS revenue
   FROM sakila_gold.fact_rental f
   JOIN sakila_gold.dim_customer c ON c.customer_id = f.customer_id
   JOIN sakila_gold.dim_film d     ON d.film_id = f.film_id
   GROUP BY c.region, d.category;

SELECT region, SUM(revenue) AS revenue FROM sakila_platinum.revenue_by_region_category GROUP BY region ORDER BY revenue DESC;

region,revenue
Asia,34986.68
Americas,19413.69
Europe,13978.85
Africa,7857.33
Oceania,863.01


### Step 5.3 · See what you built
Three charts from the tables built above:
1. **Row counts by layer, before and after the March data.** Bronze grows by everything that arrived (duplicates included), and Silver and the layers above grow only by the 2,400 new rentals.
2. **Revenue by month** from the Platinum view. The March 2006 bar is the data you salted in.
3. **Revenue by region**, which exists only because of the new country_region file.

In [37]:
import matplotlib.pyplot as plt

counts = spark.sql('''
  SELECT table_name,
         MAX(CASE WHEN snapshot = 'before' THEN row_count END) AS before,
         MAX(CASE WHEN snapshot = 'after'  THEN row_count END) AS after
  FROM workspace.default.layer_counts
  WHERE table_name IN ('sakila_bronze.rental_raw', 'sakila_silver.rental',
                       'sakila_silver_enriched.rental_enriched', 'sakila_gold.fact_rental')
  GROUP BY table_name''').toPandas()
order = ['sakila_bronze.rental_raw', 'sakila_silver.rental', 'sakila_silver_enriched.rental_enriched', 'sakila_gold.fact_rental']
counts = counts.set_index('table_name').loc[order]
labels = ['Bronze\nrental_raw', 'Silver\nrental', 'Silver-Enriched\nrental_enriched', 'Gold\nfact_rental']
monthly = spark.table("workspace.sakila_platinum.monthly_revenue").orderBy("month").toPandas()
region  = spark.sql("SELECT region, SUM(revenue) AS revenue FROM workspace.sakila_platinum.revenue_by_region_category GROUP BY region ORDER BY revenue").toPandas()

fig, ax = plt.subplots(1, 3, figsize=(18, 5))
x = range(len(order))
ax[0].bar([i - 0.2 for i in x], counts['before'], 0.4, label='before March', color='#9e9e9e')
ax[0].bar([i + 0.2 for i in x], counts['after'],  0.4, label='after March',  color='#800000')
for i, (b, a) in enumerate(zip(counts['before'], counts['after'])):
    ax[0].text(i + 0.2, a, f"+{a - b:,}", ha='center', va='bottom', fontsize=9)
ax[0].set_ylim(0, counts['after'].max() * 1.15)
ax[0].set_xticks(list(x)); ax[0].set_xticklabels(labels, fontsize=8); ax[0].set_title('Rental rows by layer'); ax[0].legend(loc='lower right')
colors = ['#800000' if m == '2006-03' else '#c9a3a3' for m in monthly['month']]
ax[1].bar(monthly['month'], monthly['revenue'].astype(float), color=colors)
ax[1].set_title('Revenue by rental month (March 2006 = the new data)'); ax[1].tick_params(axis='x', labelsize=8)
ax[2].barh(region['region'], region['revenue'].astype(float), color='#800000')
ax[2].set_title('Revenue by customer region')
plt.tight_layout(); plt.show()

[Figure: row counts by layer, revenue by month, revenue by region. Rendered as charts in this project's README.]
